# Optuna Tuning for CatBoost — Colab GPU + MLflow

**Goal:** find the best CatBoost hyperparameters for the Sparkov fraud-detection task
in ~30-50 minutes (vs ~5 hours on local CPU).

**Setup checklist (do these BEFORE running):**
1. **Runtime → Change runtime type → Hardware accelerator → GPU** (T4 is fine; A100 if Pro)
2. Upload `fraudTrain.csv` and `fraudTest.csv` to your Google Drive, e.g. to
   `MyDrive/fraud-detection/data/raw/`
3. Run cell 1 (mount Drive) — paste the auth code when prompted

**What this notebook does:**
1. Mount Drive + load data
2. Replicate the feature engineering from `FeatureEngineering`
3. Train a baseline CatBoost on GPU (sanity check, ~2 min)
4. Run Optuna trials with MLflow trial-level logging (~30-50 min)
5. Re-train with the best params (~2 min)
6. Save best params as JSON + log final model to MLflow

**Output:** `models/optuna_best.json` — used by `pipelines.training_pipeline` when `optuna.enabled=false`

In [ ]:
# ── 1. Setup ────────────────────────────────────────────────────────────
!pip install -q catboost optuna mlflow 2>&1 | tail -3

import pandas as pd
import numpy as np
import catboost as cb
import optuna
import mlflow
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.metrics import fbeta_score, f1_score, precision_score, recall_score
import time, gc, warnings, json
warnings.filterwarnings('ignore')
np.random.seed(42)
pd.set_option('display.max_columns', 100)

# ── MLflow setup ────────────────────────────────────────────────────────
# Logs to DagsHub MLflow — set DAGS_HUB_TOKEN in Colab Secrets before running
import os
from google.colab import userdata
DAGSHUB_TOKEN = userdata.get('DAGS_HUB_TOKEN') or os.getenv('DAGSHUB_TOKEN', '')
os.environ['MLFLOW_TRACKING_USERNAME'] = DAGSHUB_TOKEN
os.environ['MLFLOW_TRACKING_PASSWORD'] = DAGSHUB_TOKEN
MLFLOW_TRACKING_URI = "https://dagshub.com/NullBitZer0/real-time-fraud-detection.mlflow"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("sparkov-optuna-colab")
# Verify GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo "No GPU detected"
DEVICE = 'GPU' if __import__('subprocess').run(
    'nvidia-smi', shell=True, capture_output=True
).returncode == 0 else 'CPU'

In [ ]:
# Mount Google Drive (optional — adjust paths if using local files)
from google.colab import drive, files
drive.mount('/content/drive')

In [ ]:
# ── 2. Load data + time-based split ────────────────────────────────────
DATA_DIR = '/content/drive/MyDrive/data/fraud'
TRAIN_PATH = f'{DATA_DIR}/fraudTrain.csv'
TEST_PATH  = f'{DATA_DIR}/fraudTest.csv'

train_full = pd.read_csv(TRAIN_PATH).drop(columns=['Unnamed: 0'])
test_full  = pd.read_csv(TEST_PATH ).drop(columns=['Unnamed: 0'])
train_full['trans_date_trans_time'] = pd.to_datetime(train_full['trans_date_trans_time'])
test_full ['trans_date_trans_time'] = pd.to_datetime(test_full ['trans_date_trans_time'])

print(f"Train: {train_full.shape[0]:>9,} rows | fraud: {train_full.is_fraud.sum():>5,} ({train_full.is_fraud.mean():.4%})")
print(f"Test : {test_full.shape[0]:>9,} rows | fraud: {test_full.is_fraud.sum():>5,} ({test_full.is_fraud.mean():.4%})")

train_full = train_full.sort_values('trans_date_trans_time').reset_index(drop=True)
cutoff = train_full['trans_date_trans_time'].quantile(0.80)
val_df   = train_full[train_full['trans_date_trans_time'] >= cutoff].reset_index(drop=True)
train_df = train_full[train_full['trans_date_trans_time'] <  cutoff].reset_index(drop=True)
print(f"\nTrain: {len(train_df):>9,} | fraud: {train_df.is_fraud.sum():>4,} ({train_df.is_fraud.mean():.4%})")
print(f"Val  : {len(val_df):>9,} | fraud: {val_df.is_fraud.sum():>4,} ({val_df.is_fraud.mean():.4%})")
print(f"Test : {len(test_full):>9,} | fraud: {test_full.is_fraud.sum():>4,} ({test_full.is_fraud.mean():.4%})")

In [ ]:
# ── 3. Feature engineering (same as FeatureEngineering class) ───────────
def build_base_features(df):
    df = df.copy()
    df['hour']     = df['trans_date_trans_time'].dt.hour.astype('int8')
    df['dow']      = df['trans_date_trans_time'].dt.dayofweek.astype('int8')
    df['month']    = df['trans_date_trans_time'].dt.month.astype('int8')
    df['is_night'] = df['hour'].isin([0,1,2,3,4,22,23]).astype('int8')
    df['dob']      = pd.to_datetime(df['dob'])
    df['age']      = ((df['trans_date_trans_time'] - df['dob']).dt.days / 365.25).astype('float32')
    df['amt_log']  = np.log1p(df['amt']).astype('float32')
    df['amt_is_round'] = (df['amt'] == df['amt'].round(0)).astype('int8')
    R = 6371.0
    lat1, lon1 = np.radians(df['lat']), np.radians(df['long'])
    lat2, lon2 = np.radians(df['merch_lat']), np.radians(df['merch_long'])
    dlat = lat2 - lat1; dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    df['distance_km'] = (2 * R * np.arcsin(np.sqrt(a))).astype('float32')
    return df

train_df = build_base_features(train_df)
val_df   = build_base_features(val_df)
test_df  = build_base_features(test_full)

combined = pd.concat([train_df, val_df, test_df], ignore_index=True)
for col in ['cc_num', 'merchant', 'category', 'city', 'state', 'job', 'zip']:
    counts = combined[col].value_counts().to_dict()
    for d in (train_df, val_df, test_df):
        d[f'{col}_FE'] = d[col].map(counts).astype('float32')
del combined; gc.collect()

GLOBAL = train_df['is_fraud'].mean()
SMOOTHING = 50.0
for col in ['merchant', 'category', 'city', 'state', 'job']:
    stats = train_df.groupby(col)['is_fraud'].agg(['mean', 'count'])
    smoothed = (stats['mean'] * stats['count'] + GLOBAL * SMOOTHING) / (stats['count'] + SMOOTHING)
    for d in (train_df, val_df, test_df):
        d[f'{col}_te'] = d[col].map(smoothed.to_dict()).fillna(GLOBAL).astype('float32')

for col in ['merchant', 'category', 'cc_num']:
    stats = train_df.groupby(col)['amt'].agg(['mean', 'std'])
    for stat in ['mean', 'std']:
        mapping = stats[stat].to_dict()
        fallback = train_df['amt'].mean() if stat == 'mean' else train_df['amt'].std()
        for d in (train_df, val_df, test_df):
            d[f'amt_per_{col}_{stat}'] = d[col].map(mapping).fillna(fallback).astype('float32')

def add_velocity(df, window_hours_list):
    df = df.sort_values(['cc_num', 'trans_date_trans_time']).reset_index(drop=True)
    df['ts'] = df['trans_date_trans_time'].astype('int64') // 10**9
    for h in window_hours_list:
        window_sec = h * 3600
        df[f'txn_last_{h}h'] = (
            df.groupby('cc_num')['ts']
              .transform(lambda s: s.searchsorted(s.values - h*3600, side='right') - 1)
              .astype('float32')
        )
        amt_sums = np.zeros(len(df), dtype='float32')
        for _, g in df.groupby('cc_num', sort=False):
            ts = g['ts'].values; amt = g['amt'].values; idx = g.index.values; n = len(g)
            cum_amt = np.concatenate([[0.0], np.cumsum(amt, dtype='float64')])
            for i in range(n):
                j = np.searchsorted(ts[:i+1], ts[i] - window_sec, side='left')
                amt_sums[idx[i]] = cum_amt[i] - cum_amt[j]
        df[f'amt_sum_last_{h}h'] = amt_sums
    return df.drop(columns=['ts'])

train_df = add_velocity(train_df, [1, 24, 168])
val_df   = add_velocity(val_df,   [1, 24, 168])
test_df  = add_velocity(test_df,  [1, 24, 168])

DROP = ['trans_date_trans_time', 'first', 'last', 'street', 'dob',
        'trans_num', 'unix_time', 'lat', 'long', 'merch_lat', 'merch_long',
        'cc_num', 'merchant', 'category', 'city', 'state', 'job', 'gender', 'zip',
        'is_fraud']
FEATURES = [c for c in train_df.columns if c not in DROP]
print(f"Total features: {len(FEATURES)}")

X_train, y_train = train_df[FEATURES].values.astype('float32'), train_df['is_fraud'].values
X_val,   y_val   = val_df  [FEATURES].values.astype('float32'), val_df  ['is_fraud'].values
X_test,  y_test  = test_df [FEATURES].values.astype('float32'), test_df ['is_fraud'].values
print(f"X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}")

In [ ]:
# ── 4. Baseline CatBoost on GPU (sanity check, logged to MLflow) ────────
with mlflow.start_run(run_name="baseline") as run:
    mlflow.log_param("device", DEVICE)
    mlflow.log_param("model_type", "CatBoost")

    t0 = time.time()
    m_base = cb.CatBoostClassifier(
        iterations=500, depth=8, learning_rate=0.05,
        eval_metric='PRAUC', random_seed=42, verbose=0,
        early_stopping_rounds=50, task_type=DEVICE,
    )
    m_base.fit(X_train, y_train, eval_set=(X_val, y_val))
    t_base = time.time() - t0

    val_pr  = average_precision_score(y_val,  m_base.predict_proba(X_val)[:, 1])
    test_pr = average_precision_score(y_test, m_base.predict_proba(X_test)[:, 1])
    test_roc = roc_auc_score(y_test, m_base.predict_proba(X_test)[:, 1])

    mlflow.log_metrics({
        "val_pr_auc": val_pr,
        "test_pr_auc": test_pr,
        "test_roc_auc": test_roc,
        "fit_time_seconds": t_base,
    })
    mlflow.catboost.log_model(m_base, "catboost")

    print(f"Baseline CatBoost ({DEVICE}) | fit {t_base:.1f}s")
    print(f"  val  PR-AUC : {val_pr:.4f}")
    print(f"  test PR-AUC : {test_pr:.4f}")
    print(f"  test ROC-AUC: {test_roc:.4f}")
    print(f"  MLflow run_id: {run.info.run_id}")

In [ ]:
# ── 5. Optuna search with MLflow trial-level logging ────────────────────
N_TRIALS = 50 if DEVICE == 'GPU' else 30

print(f"Running {N_TRIALS} Optuna trials on {DEVICE}")
print(f"MLflow → {MLFLOW_TRACKING_URI}")

SEARCH_SPACE = {
    'iterations':          (300, 1500),
    'depth':               (4, 10),
    'learning_rate':       (0.01, 0.15),
    'l2_leaf_reg':         (0.5, 20.0),
    'random_strength':     (0.0, 5.0),
    'bagging_temperature': (0.0, 2.0),
    'border_count':        (32, 254),
}

with mlflow.start_run(run_name=f"optuna-{N_TRIALS}-trials") as optuna_run:
    mlflow.log_params({
        "n_trials": N_TRIALS,
        "device": DEVICE,
        "search_space": str(SEARCH_SPACE),
    })

    def objective(trial):
        params = {
            'eval_metric': 'PRAUC',
            'random_seed': 42,
            'verbose': 0,
            'early_stopping_rounds': 30,
            'task_type': DEVICE,
        }
        for param, (lo, hi) in SEARCH_SPACE.items():
            if param in ('learning_rate', 'l2_leaf_reg'):
                params[param] = trial.suggest_float(param, lo, hi, log=True)
            elif param in ('iterations', 'depth', 'border_count'):
                params[param] = trial.suggest_int(param, int(lo), int(hi))
            else:
                params[param] = trial.suggest_float(param, lo, hi)

        try:
            m = cb.CatBoostClassifier(**params)
            m.fit(X_train, y_train, eval_set=(X_val, y_val))
            pr = average_precision_score(y_val, m.predict_proba(X_val)[:, 1])
        except Exception as e:
            print(f"  Trial {trial.number} failed: {e}")
            return 0.0

        mlflow.log_metrics({f"trial_{trial.number}_pr_auc": pr}, step=trial.number)
        mlflow.log_params({f"trial_{trial.number}_{k}": v for k, v in params.items()})
        return pr

    t0 = time.time()
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=10),
    )

    def print_best(study, trial):
        if (trial.number + 1) % 5 == 0 or trial.number == 0:
            print(f"  Trial {trial.number+1:3d}/{N_TRIALS} | best: {study.best_value:.4f} | current: {trial.value:.4f} | elapsed: {time.time()-t0:.0f}s")

    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False, callbacks=[print_best])
    t_optuna = time.time() - t0

    mlflow.log_metrics({
        "optuna_best_pr_auc": study.best_value,
        "optuna_duration_min": round(t_optuna / 60, 1),
        "optuna_trials_completed": len(study.trials),
    })
    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})

    print(f"\nOptuna complete in {t_optuna/60:.1f} min")
    print(f"Best val PR-AUC: {study.best_value:.4f}")
    print(f"Best params: {study.best_params}")
    print(f"MLflow run_id: {optuna_run.info.run_id}")

In [ ]:
# ── 6. Re-train final model with best params + log to MLflow ────────────
best_params = dict(study.best_params)
best_params.update({
    'eval_metric':   'PRAUC',
    'random_seed':   42,
    'verbose':       0,
    'early_stopping_rounds': 50,
    'task_type':     DEVICE,
})

with mlflow.start_run(run_name="optuna-final") as final_run:
    mlflow.log_params(best_params)

    t0 = time.time()
    m_final = cb.CatBoostClassifier(**best_params)
    m_final.fit(X_train, y_train, eval_set=(X_val, y_val))
    t_final = time.time() - t0

    val_pr  = average_precision_score(y_val,  m_final.predict_proba(X_val)[:, 1])
    test_pr = average_precision_score(y_test, m_final.predict_proba(X_test)[:, 1])
    test_roc = roc_auc_score(y_test, m_final.predict_proba(X_test)[:, 1])

    mlflow.log_metrics({
        "val_pr_auc": val_pr,
        "test_pr_auc": test_pr,
        "test_roc_auc": test_roc,
        "fit_time_seconds": t_final,
    })
    mlflow.catboost.log_model(m_final, "catboost")

    print(f"Final CatBoost (Optuna-tuned, {DEVICE}) | fit {t_final:.1f}s")
    print(f"  val  PR-AUC : {val_pr:.4f}")
    print(f"  test PR-AUC : {test_pr:.4f}")
    print(f"  test ROC-AUC: {test_roc:.4f}")
    print(f"  MLflow run_id: {final_run.info.run_id}")

In [ ]:
# ── 7. Save best params as JSON ────────────────────────────────────────
cat_tuned = {
    'iterations':    study.best_params['iterations'],
    'depth':         study.best_params['depth'],
    'learning_rate': study.best_params['learning_rate'],
    'l2_leaf_reg':   study.best_params['l2_leaf_reg'],
    'random_strength':   study.best_params['random_strength'],
    'bagging_temperature': study.best_params['bagging_temperature'],
    'border_count':  study.best_params['border_count'],
    'eval_metric':   'PRAUC',
    'random_seed':   42,
    'verbose':       0,
    'early_stopping_rounds': 50,
    'task_type':     'CPU',
}

save_path = '/content/drive/MyDrive/data/optuna_best.json'
os.makedirs(os.path.dirname(save_path), exist_ok=True)
with open(save_path, 'w') as f:
    json.dump({"best_params": cat_tuned, "best_value": study.best_value}, f, indent=2)
print(f"Saved to Drive: {save_path}")

print("\nDownloading to your local machine...")
files.download(save_path)

print("\n" + "=" * 70)
print("Place this file at models/optuna_best.json in the project root.")
print("The training pipeline will auto-load it when optuna.enabled=false")
print("=" * 70)